# 👗 Fashion Wrapped — Personal Style Analysis
> A Spotify Wrapped-style data analysis of my 2025 wardrobe.  
> **Tools:** Python · Pandas · Matplotlib · Seaborn  
> **Data:** Personal purchase records & wear tracking logs

---
## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib import rcParams
import warnings
warnings.filterwarnings('ignore')

# ── Style config ──────────────────────────────────────────
PALETTE = {
    'ink':    '#0D0D0D',
    'cream':  '#F5F0E8',
    'sand':   '#C8B89A',
    'blush':  '#D4A5A0',
    'forest': '#2C3E2D',
    'gold':   '#B8962E',
    'smoke':  '#8A8478'
}

rcParams['font.family'] = 'serif'
rcParams['figure.facecolor'] = PALETTE['cream']
rcParams['axes.facecolor'] = PALETTE['cream']
rcParams['axes.edgecolor'] = PALETTE['smoke']
rcParams['text.color'] = PALETTE['ink']
rcParams['axes.labelcolor'] = PALETTE['ink']
rcParams['xtick.color'] = PALETTE['smoke']
rcParams['ytick.color'] = PALETTE['smoke']

print('✅ Setup complete')

---
## 1. Load & Clean Data

In [ ]:
# ── Load ──────────────────────────────────────────────────
df_raw = pd.read_csv('wardrobe_data.csv', skiprows=2)
df_raw.columns = ['num','date','brand','item_name','category',
                   'color_family','price_paid','times_worn',
                   'still_own','purchased_where','season','notes']

# ── Clean ─────────────────────────────────────────────────
df = df_raw.copy()
df = df[df['brand'].notna() & (df['brand'].str.strip() != '')]
df['price_paid']  = pd.to_numeric(df['price_paid'],  errors='coerce').fillna(0)
df['times_worn']  = pd.to_numeric(df['times_worn'],  errors='coerce').fillna(0)
df['date']        = pd.to_datetime(df['date'], errors='coerce')
df['brand']       = df['brand'].str.strip()
df['category']    = df['category'].str.strip()
df['color_family']= df['color_family'].str.strip()
df['month']       = df['date'].dt.month
df['month_name']  = df['date'].dt.strftime('%b')
df['year']        = df['date'].dt.year

# ── Derived metrics ───────────────────────────────────────
df['cost_per_wear'] = df.apply(
    lambda r: round(r['price_paid'] / r['times_worn'], 2)
    if r['times_worn'] > 0 and r['price_paid'] > 0 else np.nan, axis=1
)

print(f'✅ Loaded {len(df)} rows across {df["brand"].nunique()} brands')
df.head(10)

---
## 2. Hero KPIs

In [ ]:
total_spend    = df['price_paid'].sum()
total_pieces   = len(df)
total_wears    = df['times_worn'].sum()
avg_cost       = df[df['price_paid'] > 0]['price_paid'].mean()
cpw_overall    = total_spend / total_wears if total_wears > 0 else 0
retention_rate = (df['still_own'] == 'Yes').sum() / total_pieces * 100
unique_brands  = df['brand'].nunique()

kpis = {
    'Total Spend ($)':       f"${total_spend:,.0f}",
    'Total Pieces':          total_pieces,
    'Total Wears Logged':    int(total_wears),
    'Avg Cost / Piece ($)':  f"${avg_cost:.0f}",
    'Cost Per Wear ($)':     f"${cpw_overall:.2f}",
    'Retention Rate (%)':    f"{retention_rate:.0f}%",
    'Unique Brands':         unique_brands
}

kpi_df = pd.DataFrame(kpis.items(), columns=['Metric', 'Value'])
print('\n📊 HERO KPIs\n')
print(kpi_df.to_string(index=False))

---
## 3. Colour Palette Breakdown

In [ ]:
color_map = {
    'Black': '#1A1A18', 'Brown': '#8B6E52', 'Red': '#C0392B',
    'White': '#E8E4DC', 'Forest Green': '#2C3E2D', 'Grey': '#9E9E9E',
    'Cream': '#F5F0E8', 'Blue': '#3B5998', 'Camel': '#C8B89A',
    'Dusty Blush': '#D4A5A0', 'Multi': '#B8962E', 'Ivory': '#FFFFF0',
    'Sand': '#C2B280', 'Pink': '#FFB6C1', 'Other': '#CCCCCC'
}

color_counts = (df.groupby('color_family')
                  .agg(pieces=('price_paid','count'),
                       spend=('price_paid','sum'))
                  .sort_values('pieces', ascending=False))
color_counts['pct'] = (color_counts['pieces'] / total_pieces * 100).round(1)

top_colors = color_counts.head(8)
bar_colors = [color_map.get(c, PALETTE['sand']) for c in top_colors.index]
edge_colors = ['#888' if c in ['White','Cream','Ivory'] else 'none' for c in top_colors.index]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_colors.index[::-1], top_colors['pieces'][::-1],
               color=bar_colors[::-1], edgecolor=edge_colors[::-1],
               linewidth=0.8, height=0.6)

for bar, (idx, row) in zip(bars, top_colors[::-1].iterrows()):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f"{row['pct']}%  ({int(row['pieces'])} pieces)",
            va='center', fontsize=10, color=PALETTE['smoke'])

ax.set_xlabel('Number of Pieces', labelpad=10)
ax.set_title('My Colour Palette', fontsize=16, fontweight='bold',
             pad=20, color=PALETTE['ink'])
ax.spines[['top','right','left']].set_visible(False)
ax.xaxis.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('chart_colours.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 COLOUR BREAKDOWN')
print(color_counts[['pieces','pct','spend']].to_string())

---
## 4. Spend by Category

In [ ]:
cats = (df.groupby('category')
          .agg(spend=('price_paid','sum'),
               pieces=('price_paid','count'),
               wears=('times_worn','sum'))
          .sort_values('spend', ascending=True))
cats['cost_per_wear'] = (cats['spend'] / cats['wears'].replace(0,1)).round(2)
cats['avg_price']     = (cats['spend'] / cats['pieces']).round(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: spend by category
bar_colors = [PALETTE['gold'] if s == cats['spend'].max()
              else PALETTE['forest'] for s in cats['spend']]
axes[0].barh(cats.index, cats['spend'], color=bar_colors, height=0.6)
for i, (idx, row) in enumerate(cats.iterrows()):
    axes[0].text(row['spend'] + 5, i, f"${row['spend']:,.0f}",
                 va='center', fontsize=9, color=PALETTE['smoke'])
axes[0].set_title('Total Spend by Category', fontsize=13,
                  fontweight='bold', color=PALETTE['ink'])
axes[0].set_xlabel('Spend ($)')
axes[0].spines[['top','right','left']].set_visible(False)
axes[0].xaxis.grid(True, alpha=0.3, linestyle='--')
axes[0].set_axisbelow(True)

# Right: cost per wear by category
cpw_sorted = cats.sort_values('cost_per_wear', ascending=True)
bar_colors2 = [PALETTE['blush'] if c == cpw_sorted['cost_per_wear'].min()
               else PALETTE['sand'] for c in cpw_sorted['cost_per_wear']]
axes[1].barh(cpw_sorted.index, cpw_sorted['cost_per_wear'],
             color=bar_colors2, height=0.6)
for i, (idx, row) in enumerate(cpw_sorted.iterrows()):
    axes[1].text(row['cost_per_wear'] + 0.05, i,
                 f"${row['cost_per_wear']:.2f}/wear",
                 va='center', fontsize=9, color=PALETTE['smoke'])
axes[1].set_title('Cost Per Wear by Category', fontsize=13,
                  fontweight='bold', color=PALETTE['ink'])
axes[1].set_xlabel('Cost Per Wear ($)')
axes[1].spines[['top','right','left']].set_visible(False)
axes[1].xaxis.grid(True, alpha=0.3, linestyle='--')
axes[1].set_axisbelow(True)

plt.suptitle('Category Analysis', fontsize=16, fontweight='bold',
             color=PALETTE['ink'], y=1.02)
plt.tight_layout()
plt.savefig('chart_categories.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 CATEGORY BREAKDOWN')
print(cats[['spend','pieces','wears','cost_per_wear','avg_price']]
      .sort_values('spend', ascending=False).to_string())

---
## 5. Top Brands Analysis

In [ ]:
brands = (df.groupby('brand')
            .agg(spend=('price_paid','sum'),
                 pieces=('price_paid','count'),
                 wears=('times_worn','sum'))
            .sort_values('spend', ascending=False))
brands['cost_per_wear'] = (brands['spend'] / brands['wears'].replace(0,1)).round(2)
brands['avg_price']     = (brands['spend'] / brands['pieces']).round(0)
brands['spend_pct']     = (brands['spend'] / total_spend * 100).round(1)

top10 = brands.head(10).sort_values('spend', ascending=True)

fig, ax = plt.subplots(figsize=(11, 7))
bar_colors = [PALETTE['gold'] if b == top10['spend'].max()
              else PALETTE['forest'] for b in top10['spend']]
bars = ax.barh(top10.index, top10['spend'], color=bar_colors, height=0.65)

for bar, (idx, row) in zip(bars, top10.iterrows()):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
            f"${row['spend']:,.0f}  ·  {int(row['pieces'])} pcs  ·  ${row['cost_per_wear']:.2f}/wear",
            va='center', fontsize=8.5, color=PALETTE['smoke'])

ax.set_title('Top 10 Brands by Spend', fontsize=16,
             fontweight='bold', pad=20, color=PALETTE['ink'])
ax.set_xlabel('Total Spend ($)', labelpad=10)
ax.spines[['top','right','left']].set_visible(False)
ax.xaxis.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('chart_brands.png', dpi=150, bbox_inches='tight')
plt.show()

# Pareto analysis
top3_pct = brands['spend'].head(3).sum() / total_spend * 100
top5_pct = brands['spend'].head(5).sum() / total_spend * 100
print(f'\n📊 BRAND CONCENTRATION')
print(f'Top 3 brands = {top3_pct:.0f}% of total spend')
print(f'Top 5 brands = {top5_pct:.0f}% of total spend')
print(f'\nFull brand table:')
print(brands[['spend','pieces','wears','cost_per_wear','spend_pct']].to_string())

---
## 6. Monthly Spend Rhythm

In [ ]:
month_order = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

monthly = (df.groupby(['month','month_name'])['price_paid']
             .sum().reset_index()
             .sort_values('month'))

# Fill missing months with 0
all_months = pd.DataFrame({'month': range(1,13),
                            'month_name': month_order})
monthly = all_months.merge(monthly, on=['month','month_name'], how='left').fillna(0)

avg_monthly = monthly['price_paid'].mean()
peak_month  = monthly.loc[monthly['price_paid'].idxmax()]

fig, ax = plt.subplots(figsize=(12, 5))
bar_colors = [PALETTE['gold'] if v == monthly['price_paid'].max()
              else PALETTE['forest'] for v in monthly['price_paid']]

bars = ax.bar(monthly['month_name'], monthly['price_paid'],
              color=bar_colors, width=0.6, zorder=3)
ax.axhline(avg_monthly, color=PALETTE['blush'], linestyle='--',
           linewidth=1.5, label=f'Monthly avg: ${avg_monthly:.0f}', zorder=4)

for bar in bars:
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
                f"${bar.get_height():.0f}", ha='center',
                fontsize=8, color=PALETTE['smoke'])

ax.set_title('Monthly Spend — Where Did My Money Go?',
             fontsize=15, fontweight='bold', pad=20, color=PALETTE['ink'])
ax.set_ylabel('Spend ($)')
ax.legend(fontsize=10)
ax.spines[['top','right','left']].set_visible(False)
ax.yaxis.grid(True, alpha=0.3, linestyle='--', zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('chart_monthly.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📊 MONTHLY INSIGHTS')
print(f"Peak month:    {peak_month['month_name']} — ${peak_month['price_paid']:.0f}")
print(f"Monthly avg:   ${avg_monthly:.0f}")
print(f"Peak vs avg:   {peak_month['price_paid']/avg_monthly:.1f}x")
print(f"\nFull monthly breakdown:")
print(monthly[['month_name','price_paid']].to_string(index=False))

---
## 7. Cost-Per-Wear Deep Dive

In [ ]:
cpw_df = df[df['cost_per_wear'].notna()].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter — price vs wears
scatter_colors = [color_map.get(c, PALETTE['sand'])
                  for c in cpw_df['color_family']]
axes[0].scatter(cpw_df['times_worn'], cpw_df['price_paid'],
                c=scatter_colors, alpha=0.75, s=80, edgecolors='white',
                linewidth=0.5, zorder=3)

# Annotate top 5 best value
top5_value = cpw_df.nsmallest(5, 'cost_per_wear')
for _, row in top5_value.iterrows():
    axes[0].annotate(f"{row['brand']}\n${row['cost_per_wear']:.2f}/wear",
                     xy=(row['times_worn'], row['price_paid']),
                     xytext=(8, 8), textcoords='offset points',
                     fontsize=7, color=PALETTE['forest'],
                     arrowprops=dict(arrowstyle='->', color=PALETTE['smoke'],
                                     lw=0.8))

axes[0].set_xlabel('Times Worn')
axes[0].set_ylabel('Price Paid ($)')
axes[0].set_title('Price vs Wears\n(bottom-left = poor value, top-right = great value)',
                  fontsize=11, fontweight='bold', color=PALETTE['ink'])
axes[0].spines[['top','right']].set_visible(False)
axes[0].yaxis.grid(True, alpha=0.3, linestyle='--')
axes[0].set_axisbelow(True)

# Right: best & worst CPW items
best5  = cpw_df.nsmallest(5,  'cost_per_wear')[['brand','item_name','cost_per_wear']]
worst5 = cpw_df.nlargest(5,   'cost_per_wear')[['brand','item_name','cost_per_wear']]
combined = pd.concat([
    best5.assign(type='Best Value 🟢'),
    worst5.assign(type='Worst Value 🔴')
])
combined['label'] = combined['brand'] + '\n' + combined['item_name'].str[:20]
combined_sorted = combined.sort_values('cost_per_wear', ascending=True)
bar_c = [PALETTE['forest'] if t == 'Best Value 🟢'
         else PALETTE['blush'] for t in combined_sorted['type']]
axes[1].barh(combined_sorted['label'], combined_sorted['cost_per_wear'],
             color=bar_c, height=0.6)
for i, (_, row) in enumerate(combined_sorted.iterrows()):
    axes[1].text(row['cost_per_wear'] + 0.05, i,
                 f"${row['cost_per_wear']:.2f}",
                 va='center', fontsize=8, color=PALETTE['smoke'])
axes[1].set_title('Best vs Worst Value Pieces', fontsize=11,
                  fontweight='bold', color=PALETTE['ink'])
axes[1].set_xlabel('Cost Per Wear ($)')
axes[1].spines[['top','right','left']].set_visible(False)
axes[1].xaxis.grid(True, alpha=0.3, linestyle='--')
axes[1].set_axisbelow(True)
green_patch = mpatches.Patch(color=PALETTE['forest'], label='Best Value')
red_patch   = mpatches.Patch(color=PALETTE['blush'],  label='Worst Value')
axes[1].legend(handles=[green_patch, red_patch], fontsize=9)

plt.suptitle('Cost-Per-Wear Analysis', fontsize=15,
             fontweight='bold', color=PALETTE['ink'], y=1.02)
plt.tight_layout()
plt.savefig('chart_cpw.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 TOP 5 BEST VALUE PIECES')
print(best5.to_string(index=False))
print('\n📊 TOP 5 WORST VALUE PIECES')
print(worst5.to_string(index=False))

---
## 8. Sustainability Score

In [ ]:
retention_rate  = (df['still_own'] == 'Yes').sum() / total_pieces * 100
gifted_pct      = df['purchased_where'].str.lower().eq('gift').sum() / total_pieces * 100
secondhand_pct  = df['purchased_where'].str.lower().eq('secondhand').sum() / total_pieces * 100
sample_sale_pct = df['purchased_where'].str.lower().eq('sample sale').sum() / total_pieces * 100
sustainable_pct = gifted_pct + secondhand_pct + sample_sale_pct

# Avg wears vs industry benchmark (industry avg ~30 wears before discard)
INDUSTRY_BENCH  = 30
avg_wears       = df[df['times_worn'] > 0]['times_worn'].mean()
wear_vs_bench   = avg_wears / INDUSTRY_BENCH * 100

metrics = {
    'Retention Rate':           f"{retention_rate:.0f}%",
    'Non-new Purchases':        f"{sustainable_pct:.0f}%",
    'Avg Wears Per Item':       f"{avg_wears:.0f}",
    'vs Industry Avg (30)':     f"{wear_vs_bench:.0f}%",
    'Overall CPW':              f"${cpw_overall:.2f}"
}

fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')
table_data = [[k, v] for k, v in metrics.items()]
table = ax.table(cellText=table_data,
                 colLabels=['Sustainability Metric', 'Value'],
                 cellLoc='left', loc='center',
                 colWidths=[0.6, 0.3])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2)
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor(PALETTE['forest'])
        cell.set_text_props(color='white', fontweight='bold')
    else:
        cell.set_facecolor(PALETTE['cream'] if r % 2 == 0 else 'white')
    cell.set_edgecolor(PALETTE['sand'])
ax.set_title('Sustainability Metrics', fontsize=14,
             fontweight='bold', color=PALETTE['ink'], pad=20)
plt.tight_layout()
plt.savefig('chart_sustainability.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 SUSTAINABILITY SUMMARY')
for k, v in metrics.items():
    print(f'  {k}: {v}')

---
## 9. Style Archetype — Scoring

In [ ]:
neutrals = ['Black','White','Cream','Ivory','Camel','Sand','Beige','Grey']
neutral_pct      = df['color_family'].isin(neutrals).sum() / total_pieces * 100
investment_pct   = (df['price_paid'] > 100).sum() / total_pieces * 100
top_brand_pieces = df['brand'].value_counts().iloc[0] / total_pieces * 100
cpw_score        = min(100, (10 / cpw_overall) * 100) if cpw_overall > 0 else 0

traits = {
    'Wardrobe Retention':        round(retention_rate),
    'Cost-Per-Wear Efficiency':  round(cpw_score),
    'Brand Exploration':         round(min(100, unique_brands / 30 * 100)),
    'Neutral Palette Discipline':round(neutral_pct),
    'Investment vs Impulse':     round(investment_pct)
}

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(list(traits.keys()), list(traits.values()),
               color=PALETTE['forest'], height=0.55)
for bar, val in zip(bars, traits.values()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', fontsize=10,
            color=PALETTE['ink'], fontweight='bold')
ax.set_xlim(0, 115)
ax.set_xlabel('Score (%)')
ax.set_title('Style Archetype — The Bold Curator',
             fontsize=14, fontweight='bold', color=PALETTE['ink'], pad=15)
ax.spines[['top','right','left']].set_visible(False)
ax.xaxis.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('chart_archetype.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 ARCHETYPE SCORES')
for k, v in traits.items():
    print(f'  {k}: {v}%')

---
## 10. Key Takeaways

In [ ]:
best_cpw  = cpw_df.loc[cpw_df['cost_per_wear'].idxmin()]
worst_cpw = cpw_df.loc[cpw_df['cost_per_wear'].idxmax()]
top_brand = brands.index[0]
top_cat   = cats.sort_values('spend', ascending=False).index[0]
best_cat_val = cats.sort_values('cost_per_wear').index[0]

print('=' * 55)
print('        FASHION WRAPPED — KEY TAKEAWAYS')
print('=' * 55)
print(f'  💰 I spent ${total_spend:,.0f} across {total_pieces} pieces')
print(f'  👗 My overall cost-per-wear: ${cpw_overall:.2f}')
print(f'  🏆 Top brand by spend: {top_brand}')
print(f'  📦 Biggest category: {top_cat}')
print(f'  ✅ Best value: {best_cpw["brand"]} {best_cpw["item_name"][:25]}')
print(f'     → ${best_cpw["cost_per_wear"]:.2f}/wear ({int(best_cpw["times_worn"])} wears)')
print(f'  ⚠️  Worst value: {worst_cpw["brand"]} {worst_cpw["item_name"][:25]}')
print(f'     → ${worst_cpw["cost_per_wear"]:.2f}/wear ({int(worst_cpw["times_worn"])} wears)')
print(f'  ♻️  Retention rate: {retention_rate:.0f}% still owned')
print(f'  🎨 Dominant colour: {color_counts.index[0]} ({color_counts["pct"].iloc[0]}%)')
print(f'  📅 Peak spend month: {peak_month["month_name"]} (${peak_month["price_paid"]:.0f})')
print('=' * 55)